In [1]:
!pip install srt

  Preparing metadata (setup.py) ... done
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22427 sha256=213c95be88c47cc54d7593ec32c4887fa54b021efed3d264deb28c8c96c817e6
  Stored in directory: /root/.cache/pip/wheels/7e/75/5b/e1d5c3756631e4bda806f6cc9640153b39484bb6f7b0b8def3
Successfully built srt


In [ ]:
from google.colab import files

uploaded = files.upload()

In [2]:
import os
import csv
import re
import subprocess
import zipfile
from pathlib import Path

import srt

# CONFIGURACIÓN

AUDIO_FILE = "A1_A_good_nights_sleep.mp3"
SRT_FILE = "A1_A_good_nights_sleep.srt"

OUTPUT_DIR = "output_files"

# margen para no cortar palabras
PRE_PADDING = 0.10
POST_PADDING = 0.20

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Leer SRT

with open(SRT_FILE, "r", encoding="utf-8-sig") as f:
    subtitles = list(srt.parse(f.read()))

audio_base_name = Path(AUDIO_FILE).stem

tsv_name = f"{audio_base_name}_anki.tsv"

with open(tsv_name, "w", newline="", encoding="utf-8") as tsv:

    writer = csv.writer(tsv, delimiter="\t")

    writer.writerow([
        "Filename",
        "AudioTag",
        "Sentence"
    ])

    total = len(subtitles)

    for i, sub in enumerate(subtitles, start=1):

        start_time = max(
            0,
            sub.start.total_seconds() - PRE_PADDING
        )

        end_time = (
            sub.end.total_seconds()
            + POST_PADDING
        )

        duration = end_time - start_time

        filename = (
            f"{audio_base_name}"
            f"_Line_{i:04d}.mp3"
        )

        output_mp3 = os.path.join(
            OUTPUT_DIR,
            filename
        )

        cmd = [
            "ffmpeg",
            "-loglevel",
            "error",
            "-y",
            "-i",
            AUDIO_FILE,
            "-ss",
            str(start_time),
            "-t",
            str(duration),
            "-acodec",
            "libmp3lame",
            output_mp3
        ]

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            print(
                f"Error en línea {i}"
            )
            print(result.stderr)
            continue

        sentence = sub.content

        # eliminar etiquetas html
        sentence = re.sub(
            r"<[^>]+>",
            "",
            sentence
        )

        sentence = (
            sentence
            .replace("\n", " ")
            .strip()
        )

        writer.writerow([
            filename,
            f"[sound:{filename}]",
            sentence
        ])

        if i % 25 == 0:
            print(
                f"{i}/{total} procesados"
            )

print("Finalizado")

25/34 procesados
Finalizado


In [3]:
zip_name = "anki_audio_package.zip"

with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files_ in os.walk(
        "output_files"
    ):
        for file in files_:
            path = os.path.join(
                root,
                file
            )

            zipf.write(
                path,
                os.path.relpath(
                    path,
                    "output_files"
                )
            )

    zipf.write(
        "audio_anki.tsv"
        if os.path.exists("audio_anki.tsv")
        else tsv_name
    )

print("ZIP creado")

ZIP creado


In [4]:
from google.colab import files

files.download("anki_audio_package.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>